# 14 — Torch button: why was this movie recommended? (Step 11)

**The problem.** Each recommendation needs a short "why" that is true. An LLM writing freely would invent pleasant reasons that have nothing to do with the ranker's real choice.

**Three layers, each tied to the actual score**
1. **SHAP:** how much each of the ranker's 8 signals pushed this movie up or down for this user. LightGBM computes exact TreeSHAP itself, so no extra library is needed.
2. **Evidence behind the signals.** Both splits are exact: the parts add up to the score.
   - *Taste match (ALS):* which of the user's liked movies contribute most to this movie's ALS score.
   - *Vibe match:* which vibes contribute most to the movie-to-taste cosine.
   - *Popularity, rating, year:* the values themselves.
3. **Wording:** a free fill-in-the-blank sentence, or Claude seeing only that evidence.

**Tests (roadmap gate: explanations match the real score drivers)**
1. **Is SHAP faithful to our ranker?** Set a movie's top-SHAP signal to an ordinary value (the median over that user's 300 candidates) and see how far it falls. It should fall further than when the most important signal overall is removed, or a random one.
2. **Does Claude stay inside the evidence?** An automatic check flags any movie or vibe in its sentence that was not in the evidence.
3. **Is Claude worth paying for?** Adil compares 20 pairs blind (template vs Claude): which is clearer, and is either wrong. Claude stays only if it wins.

Budget: about 200 Claude calls, roughly $0.20–0.30.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import lightgbm as lgb
import numpy as np
import pandas as pd

ROOT = Path("..")
sys.path.insert(0, str(ROOT.resolve()))
from src.viberec import evaluation as ev, explain as ex, llm_budget as lb, llm_parse as lp, ranker as rk

P, A, M = ROOT / "data" / "processed", ROOT / "artifacts", ROOT / "models"
VERSION, SEED, K_SHRINK = "v1", 42, 7.324
rng = np.random.default_rng(SEED)


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/14_torch.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


# the same test split and movie table as notebook 12
pre2018 = pd.read_parquet(P / f"ratings_before_2018_{VERSION}.parquet")
early = pd.read_parquet(A / f"q43_early_counts_{VERSION}.parquet")
VAL_CUT = int(early.loc[early["quantile"] == 0.8, "cutoff_ts"].iloc[0])
TEST_CUT = int(early.loc[early["quantile"] == 0.9, "cutoff_ts"].iloc[0])
movies = pd.read_parquet(P / "movies_full.parquet")
movies["genres"] = movies["genres"].str.replace(",", "|", regex=False)
tiers = pd.read_parquet(P / "movie_tiers_v2.parquet").set_index("movieId")
known = ev.movies_known_at(early[early["quantile"] == 0.9], movies, K_SHRINK, pd.to_datetime(TEST_CUT, unit="s").year)
recent = rk.recent_counts(pre2018, TEST_CUT, TEST_CUT - VAL_CUT)
mt = rk.movie_table(known, recent, tiers["year"])
train = pd.read_parquet(P / f"benchmark_train_{VERSION}.parquet")
user_mean = train.groupby("userId")["rating"].mean()
hits = pd.read_parquet(P / f"benchmark_hits_{VERSION}.parquet").groupby("userId")["movieId"].apply(set).to_dict()
titles = pd.read_csv(ROOT / "data" / "raw" / "movielens" / "ml-25m" / "movies.csv").set_index("movieId")["title"]
val_rows = pd.read_parquet(A / f"ranker_rows_val_{VERSION}.parquet").fillna(0).sort_values(["userId", "blend_rank"]).reset_index(drop=True)
test_rows = pd.read_parquet(A / f"ranker_rows_test_{VERSION}.parquet").fillna(0).sort_values(["userId", "blend_rank"]).reset_index(drop=True)
print(f"test: {test_rows['userId'].nunique():,} users x 300 candidates; features {rk.FEATURES}")

test: 5,000 users x 300 candidates; features ['als_z', 'vibe_z', 'blend_rank', 'shrunk', 'tier', 'log_count', 'log_recent', 'year_gap']


## 1. The ranker, saved

Notebook 12 picked LightGBM (the tuned settings are in `artifacts/ranker_tuning_v1.csv`) but never saved the model. It is retrained here the same way, on the 2016 validation rows, and saved. The test recall@10 must match notebook 12's 0.0764, which shows it is the same model.

In [2]:
MODEL_PATH = M / f"ranker_lgbm_{VERSION}.txt"
tuned = pd.read_csv(A / f"ranker_tuning_{VERSION}.csv")
best = json.loads(tuned[tuned["kind"] == "lightgbm"].sort_values(["recall@10", "recall@50"], ascending=False)["params"].iloc[0])
if MODEL_PATH.exists():
    print(f"{MODEL_PATH} already exists, loading.")
    booster = lgb.Booster(model_file=str(MODEL_PATH))
else:
    start = time.time()
    model = lgb.LGBMRanker(objective="lambdarank", n_estimators=best["trees"], num_leaves=best["leaves"], learning_rate=0.05,
                           min_child_samples=50, random_state=SEED, verbose=-1)
    model.fit(val_rows[rk.FEATURES].to_numpy(), val_rows["label"].to_numpy(),
              group=val_rows.groupby("userId", sort=False).size().to_numpy())
    booster = model.booster_
    booster.save_model(str(MODEL_PATH))
    write_metadata(MODEL_PATH, params=best, features=rk.FEATURES, trained_on="artifacts/ranker_rows_val_v1.parquet",
                   runtime_seconds=round(time.time() - start, 1))
test_rows["score"] = booster.predict(test_rows[rk.FEATURES].to_numpy())
recs = rk.top_k_by(test_rows, test_rows["score"].to_numpy(), k=10)
print(f"params {best}; test recall@10 {np.mean([ev.recall_at_k(recs.get(u, []), h, 10) for u, h in hits.items()]):.4f} "
      f"(notebook 12: 0.0764)")

../models/ranker_lgbm_v1.txt already exists, loading.


params {'leaves': 15, 'trees': 600}; test recall@10 0.0764 (notebook 12: 0.0764)


## 2. SHAP: which signals drive each recommendation

For every user's top 10, SHAP splits the movie's score into one part per signal. The **top driver** is the signal that pushed the movie up the most. If one signal is the top driver almost everywhere, explanations will all sound alike.

In [3]:
top10 = test_rows.assign(rank=test_rows.groupby("userId")["score"].rank(ascending=False, method="first"))
top10 = top10[top10["rank"] <= 10].reset_index(drop=True)
shap = ex.shap_frame(booster, top10[rk.FEATURES], rk.FEATURES)
assert np.allclose(shap.sum(axis=1), top10["score"], atol=1e-6)              # SHAP parts add up to the score
top10["driver1"] = [(ex.top_drivers(r, 1) or ["none"])[0] for _, r in shap.iterrows()]
top10["driver2"] = [(ex.top_drivers(r, 2) + ["none", "none"])[1] for _, r in shap.iterrows()]
print("share of top-10 recommendations by top driver:")
print(top10["driver1"].value_counts(normalize=True).round(3).to_string())
print("\nmean |SHAP| per signal (overall importance):")
print(shap[rk.FEATURES].abs().mean().sort_values(ascending=False).round(3).to_string())

share of top-10 recommendations by top driver:
driver1
log_recent    0.734
log_count     0.153
als_z         0.102
blend_rank    0.010
vibe_z        0.000
shrunk        0.000

mean |SHAP| per signal (overall importance):
log_recent    0.972
blend_rank    0.471
als_z         0.440
log_count     0.405
shrunk        0.127
year_gap      0.092
vibe_z        0.077
tier          0.000


## Test 1: is SHAP faithful? (deletion test)

For 500 random users and each of their top-10 movies, one signal at a time is set to an ordinary value (the median over that user's 300 candidates), and we measure how many places the movie falls. Three signals are compared for the same movie:
- the movie's own **top SHAP driver** (what the Torch button would say);
- the **most important signal overall** (one explanation for every movie);
- a **random** other signal.

If SHAP is faithful, removing its driver should hurt the most. The intervals come from resampling users.

In [4]:
FEATS = [f for f in rk.FEATURES if f != "tier"]                         # tier is 3 for every candidate
global_top = shap[FEATS].abs().mean().idxmax()
users = rng.choice(test_rows["userId"].unique(), 500, replace=False)
by_user = {u: g for u, g in test_rows[test_rows["userId"].isin(users)].groupby("userId")}
recs_d = []
for u in users:
    g = by_user[u]
    X, s = g[rk.FEATURES].to_numpy(), g["score"].to_numpy()
    med = np.median(X, axis=0)
    order = np.argsort(-s)[:10]
    sh = ex.shap_frame(booster, X[order], rk.FEATURES)
    mods, tags = [], []
    for k, i in enumerate(order):
        d1 = (ex.top_drivers(sh.iloc[k][FEATS], 1) or [global_top])[0]
        rand = rng.choice([f for f in FEATS if f != d1])
        for kind, f in (("shap_top", d1), ("global_top", global_top), ("random", rand)):
            x = X[i].copy(); x[rk.FEATURES.index(f)] = med[rk.FEATURES.index(f)]
            mods.append(x); tags.append((u, k + 1, kind, f, i))
    new = booster.predict(np.array(mods))
    for (u_, old_rank, kind, f, i), ns in zip(tags, new):
        others = np.delete(s, i)
        recs_d.append({"userId": u_, "old_rank": old_rank, "kind": kind, "feature": f,
                       "new_rank": int(1 + (others > ns).sum()), "drop": int(1 + (others > ns).sum()) - old_rank})
deletion = pd.DataFrame(recs_d)
per_user = deletion.groupby(["userId", "kind"])["drop"].mean().unstack()
print(f"most important signal overall: {global_top}")
print("average places fallen when the signal is set to an ordinary value:")
print(deletion.groupby("kind")["drop"].agg(["mean", "median"]).round(1).to_string())
print("share of movies that leave the top 10:", deletion.assign(out=deletion["new_rank"] > 10).groupby("kind")["out"].mean().round(3).to_dict())
for other in ("global_top", "random"):
    d = (per_user["shap_top"] - per_user[other]).to_numpy()
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    print(f"shap_top minus {other}: {d.mean():+.1f} places [{np.percentile(boot, 2.5):+.1f}, {np.percentile(boot, 97.5):+.1f}]")
same = (deletion[deletion["kind"] == "shap_top"]["feature"].to_numpy() == global_top).mean()
print(f"(the top SHAP driver IS the overall-top signal for {same:.0%} of movies; there the two rows are identical)")
deletion.to_csv(A / "torch_deletion_test_v1.csv", index=False)

most important signal overall: log_recent
average places fallen when the signal is set to an ordinary value:
            mean  median
kind                    
global_top  44.1    23.0
random       8.6     1.0
shap_top    56.3    35.0
share of movies that leave the top 10: {'global_top': 0.804, 'random': 0.348, 'shap_top': 0.89}
shap_top minus global_top: +12.2 places [+10.5, +14.0]
shap_top minus random: +47.7 places [+44.9, +50.4]
(the top SHAP driver IS the overall-top signal for 75% of movies; there the two rows are identical)


## 3. Evidence behind the drivers

**Because you liked…** The ALS score of a movie splits exactly into one part per movie the user liked (`explain.liked_contributions`, the same formula as onboarding's fold-in). The check below confirms that fold-in reproduces the saved user factors. Without that, the "because you liked" movies would describe a different score.

**Shared vibes:** the vibes contributing most to the cosine between the movie's vibe profile and the user's vibe taste.

In [5]:
from src.viberec import recommenders as rc

itf = np.load(M / f"als_item_factors_{VERSION}.npy")
uf = np.load(M / f"als_user_factors_{VERSION}.npy")
als_cols = pd.Series(np.arange(len(itf)), index=pd.read_parquet(M / f"als_movie_index_{VERSION}.parquet")["movieId"])
als_users = pd.Series(np.arange(len(uf)), index=pd.read_parquet(M / f"als_user_index_{VERSION}.parquet")["userId"])
cf_tuning = pd.read_csv(A / f"cf_tuning_{VERSION}.csv")
best_als = json.loads(cf_tuning[cf_tuning["method"] == "als"].sort_values(["recall@10", "recall@50"], ascending=False)["params"].iloc[0])
ALPHA, REG = float(best_als["alpha"]), float(best_als["reg"])
yty = itf.T @ itf
by_user_train = dict(tuple(train.groupby("userId")))


def liked_ids(u):
    r = by_user_train[u]
    return [m for m in r.loc[r["rating"] > user_mean[u], "movieId"] if m in als_cols.index]


# fold-in check: does the exact split describe the same scores as the saved user factors?
from src.viberec import cf
check = []
for u in users[:100]:
    cols = als_cols.loc[liked_ids(u)].to_numpy()
    folded = cf.fold_in_user(itf, yty, cols, ALPHA, REG)
    cand = als_cols.reindex(by_user[u]["movieId"]).dropna().astype(int).to_numpy()
    check.append(np.corrcoef(itf[cand] @ folded, itf[cand] @ uf[als_users[u]])[0, 1])
print(f"ALS {best_als}; fold-in vs saved user factors over each user's 300 candidates: "
      f"median correlation {np.median(check):.3f} (min {np.min(check):.3f})")

vibes = pd.read_parquet(P / "vibe_scores_v2.parquet").set_index("movieId").drop(columns="source").astype(float)
vz = rc.standardise(vibes)


def evidence(u, m):
    liked = liked_ids(u)
    parts = pd.Series(ex.liked_contributions(itf, yty, als_cols.loc[liked].to_numpy(), als_cols[m], ALPHA, REG), index=liked) \
        if liked and m in als_cols.index else pd.Series(dtype=float)
    liked_v = [x for x in liked if x in vz.index]
    shared = ex.top_positive(ex.vibe_contributions(vz.loc[m], vz.loc[liked_v].mean()), 3) if liked_v and m in vz.index else []
    return {"because_liked": [ex.display_title(titles[x]) for x in ex.top_positive(parts, 2)],
            "shared_vibes": [ex.VIBE_WORDS[a] for a in shared],
            "recent_ratings": int(round(np.expm1(mt.at[m, "log_recent"]))), "fair_score": float(mt.at[m, "shrunk"]),
            "year": int(tiers.at[m, "year"]), "all_ratings": int(mt.at[m, "rating_count"])}


u0 = users[0]
for _, r in top10[top10["userId"] == u0].head(3).iterrows():
    e = evidence(u0, r["movieId"])
    print(f"\n{titles[r['movieId']]} | drivers {r['driver1']}, {r['driver2']}")
    print("  " + ex.template_sentence([r["driver1"], r["driver2"]], e))

ALS {'alpha': 10.0, 'factors': 128, 'reg': 0.1, 'table': 'liked'}; fold-in vs saved user factors over each user's 300 candidates: median correlation 0.999 (min 0.997)

Whiplash (2014) | drivers log_recent, blend_rank
  Recommended because it is popular right now (3,132 ratings in the last 18 months), and people who liked Gone Girl (2014) and The Imitation Game (2014) also liked it.

American History X (1998) | drivers log_recent, blend_rank
  Recommended because it is popular right now (3,502 ratings in the last 18 months), and people who liked American Beauty (1999) and Requiem for a Dream (2000) also liked it.

Pan's Labyrinth (Laberinto del fauno, El) (2006) | drivers log_recent, blend_rank
  Recommended because it is popular right now (3,066 ratings in the last 18 months), and people who liked Children of Men (2006) and Little Miss Sunshine (2006) also liked it.


## 4. Wording: free template vs Claude, and the grounding check

One recommendation per user, for 200 random users (picked at random from each user's top 10). Each gets:
- the **template sentence** (free);
- **Claude's sentence**, from the same top-2 drivers and evidence only.

Then the automatic grounding check runs on Claude's sentences. It looks for any catalog title (the 5,000 most-rated movies), vibe word or genre name that was not in the evidence. Claude is told not to describe the movie itself; the first wording run padded sentences with \"this thriller\", \"gripping storytelling\" and similar.

In [6]:
guard = lb.BudgetGuard(A / "llm_spend_ledger.json", cap_usd=15.0)
client = boto3.Session(profile_name="viberec", region_name="us-east-2").client("bedrock-runtime")
TORCH_CACHE = ROOT / "data" / "raw" / "bedrock" / "torch_v1"
catalog = list(titles.reindex(mt["rating_count"].sort_values(ascending=False).index[:5000]).dropna())
print(f"LLM spend before: ${guard.spent():.4f}")

sample_users = rng.choice(test_rows["userId"].unique(), 200, replace=False)
rows_w = []
for u in sample_users:
    r = top10[top10["userId"] == u].sample(1, random_state=int(u)).iloc[0]
    m, drivers = int(r["movieId"]), [d for d in (r["driver1"], r["driver2"]) if d != "none"]
    e = evidence(u, m)
    evid = ex.evidence_for_llm(ex.display_title(titles[m]), drivers, e)
    text = ex.llm_wording(client, guard, lp.MODEL_ID, evid, TORCH_CACHE)
    allowed_titles = [ex.display_title(titles[m])] + sum([x.get("you_liked", []) for x in evid["reasons"]], [])
    allowed_vibes = [a for a, w in ex.VIBE_WORDS.items() if w in sum([x.get("shared_feel", []) for x in evid["reasons"]], [])]
    rows_w.append({"userId": u, "movieId": m, "movie": ex.display_title(titles[m]), "drivers": ", ".join(drivers),
                   "template": ex.template_sentence(drivers, e), "claude": text,
                   "problems": "; ".join(ex.grounding_problems(text, allowed_titles, allowed_vibes, catalog))})
wording = pd.DataFrame(rows_w)
wording.to_csv(A / "torch_wording_v3.csv", index=False)
print(f"LLM spend after: ${guard.spent():.4f}")
print(f"Claude sentences with an ungrounded movie or vibe: {(wording['problems'] != '').sum()} of {len(wording)}")
for r in wording[wording["problems"] != ""].itertuples():
    print(f"- {r.movie}: {r.claude}\n    flagged: {r.problems}")
print("\nwords per sentence: template", wording["template"].str.split().str.len().median(),
      "| Claude", wording["claude"].str.split().str.len().median())
for r in wording.head(5).itertuples():
    print(f"\n{r.movie} [{r.drivers}]\n  template: {r.template}\n  Claude:   {r.claude}")

LLM spend before: $9.0654


LLM spend after: $9.0654
Claude sentences with an ungrounded movie or vibe: 4 of 200
- Hidden Figures (2016): This movie has been getting lots of ratings recently and many people have already watched it, so you'd be joining a conversation that's happening right now.
    flagged: title: Conversation, The (1974); title: Happening, The (2008)
- Guardians of the Galaxy (2014): This one's getting a lot of attention lately, and since you enjoyed Star Wars: Episode VII and Big Hero 6, you'll probably love it too!
    flagged: genre: war
- The Reader (2008): Since you loved The Nun's Story, people with your taste really enjoyed this one. You'll appreciate its tragic, melancholic, and slow-paced approach to storytelling.
    flagged: title: Storytelling (2001)
- The Simpsons Movie (2007): People who loved Borat and Superbad really enjoyed this one, so it seems like a perfect fit for your taste in comedy!
    flagged: genre: comedy

words per sentence: template 28.0 | Claude 26.0

Hidden Figures

## Test 3: Adil's blind comparison sheet

There are 20 of the 200 pairs, picked at random. Each row shows the user's five highest-rated movies (so the reasons can be judged), the recommended movie, and two explanations in random order as A and B. Which one is Claude's is kept in a separate key file and revealed after rating.

Sheet: `data/labels/torch_explanations_v3.xlsx`.

In [7]:
SHEET = ROOT / "data" / "labels" / "torch_explanations_v3.xlsx"
KEY = ROOT / "data" / "labels" / "torch_explanations_v3_key.parquet"
if SHEET.exists():
    print(f"{SHEET} already exists; not overwritten (it may hold Adil's answers).")
else:
    pick = wording.sample(20, random_state=SEED).reset_index(drop=True)
    claude_is_a = rng.random(20) < 0.5
    sheet = pd.DataFrame({
        "#": np.arange(1, 21),
        "This user's 5 highest-rated movies": [" · ".join(ex.display_title(t) for t in titles.reindex(by_user_train[u].sort_values("rating", ascending=False)["movieId"].head(5)))
                                              for u in pick["userId"]],
        "Recommended movie": pick["movie"],
        "Explanation A": np.where(claude_is_a, pick["claude"], pick["template"]),
        "Explanation B": np.where(claude_is_a, pick["template"], pick["claude"]),
        "Which is clearer? (A / B / same)": "", "Is either wrong or misleading? (A / B / both / neither)": "", "Notes": ""})
    howto = pd.DataFrame({"How to fill this in": [
        "Each row is a movie recommended to a MovieLens user. Their five favourite movies are shown so you can judge the reasons.",
        "Read explanations A and B. One is a fill-in-the-blank sentence, the other was written by Claude. Which is which is hidden.",
        "Column F: which explanation is clearer and more convincing? A, B or same.",
        "Column G: is either explanation wrong or misleading, given the user's favourites? A, B, both or neither.",
        "Notes are optional. Save the file when done."]})
    with pd.ExcelWriter(SHEET) as w:
        howto.to_excel(w, sheet_name="How to", index=False)
        sheet.to_excel(w, sheet_name="Explanations", index=False)
    pick.assign(claude_is=np.where(claude_is_a, "A", "B"))[["userId", "movieId", "claude_is"]].to_parquet(KEY, index=False)
    print(f"saved {SHEET} and the key {KEY}")

../data/labels/torch_explanations_v3.xlsx already exists; not overwritten (it may hold Adil's answers).


## Results so far (2026-10-10)

| Check | Result |
|---|---|
| Ranker saved and identical to notebook 12 | recall@10 0.0764 = 0.0764; `models/ranker_lgbm_v1.txt` |
| **Test 1: SHAP is faithful** | Setting a movie's top-SHAP signal to an ordinary value drops it **56 places** on average (89% leave the top 10). The overall-top signal drops it 44 places (80%), a random one 9 (35%). SHAP's driver beats the overall-top signal by +12 places [+10.5, +14.0] and a random one by +48 [+45, +50]. |
| Because-you-liked evidence is exact | fold-in reproduces the saved ALS user factors: median correlation 0.999 (min 0.997) |
| **Test 2: Claude stays inside the evidence** | After instructing it not to describe the movie: 4 of 200 sentences flagged automatically. Read by hand, 1 is real ("your taste in comedy"); 3 are false alarms ("joining a conversation", "Star Wars" read as the genre War, a vibe that was in the evidence). The first wording run, without that instruction, padded sentences with unsupported description ("this thriller", "gripping storytelling"). |
| Test 3: is Claude worth paying for? | Waiting for Adil's blind ratings: `data/labels/torch_explanations_v3.xlsx` |
| Cost | $0.15 for three wording runs (600 sentences); total spend $9.07 of $20 |

**What the explanations reveal about the ranker:** the top driver is present-day popularity for 73% of recommendations, all-time popularity for 15%, taste match (ALS) for 10%, and **vibe match for none**. So an honest Torch button mostly says "popular right now, and people with your taste liked it". The ranker's vibe signal is real (Step 7 showed the blend helps), but inside the LightGBM ranker it is rarely the deciding push. The Torch button is honest about that; making vibe count more is a ranking question, not an explanation question.